# Chapter 5 - Merge Baseline HLL++ (omogeneo)

Notebook per analizzare la baseline omogenea del merge di `HyperLogLog++` sui namespace `merge_hpp_hom_d_*`.

Obiettivi:
- caricare tutti i CSV `results_merge.csv` della matrice minima;
- ricostruire `d`, `rho`, `hash` e `k` dal path dei file;
- costruire una tabella di sintesi per cella sperimentale;
- salvare figure compatte per verificare che `merge ~= serial` nel baseline omogeneo.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Repository root non trovata: esegui il notebook dalla repo o una sua sottocartella')


REPO = find_repo_root(Path.cwd().resolve())
RESULTS_ROOT = REPO / 'results'
NAMESPACE_GLOB = 'merge_hpp_hom_d_*'
RESULTS_GLOB = f'{NAMESPACE_GLOB}/merge/hllpp/*/k_*/results_merge.csv'
OUT_DIR = REPO / 'thesis' / 'figures' / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

HASH_ORDER = ['splitmix64', 'xxhash64', 'murmurhash3', 'siphash24']

print('REPO:', REPO)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('OUT_DIR:', OUT_DIR)
print('RESULTS_GLOB:', RESULTS_GLOB)


REPO: /home/daniele/Dev/satp-cpp
RESULTS_ROOT: /home/daniele/Dev/satp-cpp/results
OUT_DIR: /home/daniele/Dev/satp-cpp/thesis/figures/results
RESULTS_GLOB: merge_hpp_hom_d_*/merge/hllpp/*/k_*/results_merge.csv


In [2]:
paths = sorted(RESULTS_ROOT.glob(RESULTS_GLOB))
if not paths:
    raise FileNotFoundError(f'Nessun CSV trovato per il pattern: {RESULTS_GLOB}')


def parse_metadata(path: Path) -> dict[str, object]:
    relative = path.relative_to(REPO)
    parts = relative.parts
    namespace = parts[1]
    hash_name = parts[4]
    k_dir = parts[5]

    ns_match = re.fullmatch(r'merge_hpp_hom_d_(\d+)', namespace)
    if ns_match is None:
        raise ValueError(f'Namespace inatteso: {namespace}')
    d = int(ns_match.group(1))

    k_match = re.fullmatch(r'k_(\d+)', k_dir)
    if k_match is None:
        raise ValueError(f'Directory k inattesa: {k_dir}')
    k = int(k_match.group(1))

    return {
        'namespace': namespace,
        'hash': hash_name,
        'k': k,
        'd': d,
    }


frames = []
for path in paths:
    meta = parse_metadata(path)
    df = pd.read_csv(path)
    for key, value in meta.items():
        df[key] = value
    df['source_path'] = str(path)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

numeric_cols = [
    'pairs',
    'sample_size',
    'pair_index',
    'seed',
    'estimate_merge',
    'estimate_serial',
    'delta_merge_serial_abs',
    'delta_merge_serial_rel',
    'k',
    'd',
]
for col in numeric_cols:
    raw[col] = pd.to_numeric(raw[col])

raw['rho'] = (raw['sample_size'] / raw['d']).astype(int)

print('files:', len(paths))
print('rows:', len(raw))
print('hashes:', sorted(raw['hash'].unique().tolist()))
print('k values:', sorted(raw['k'].unique().tolist()))
print('d values:', sorted(raw['d'].unique().tolist()))
print('rho values:', sorted(raw['rho'].unique().tolist()))

display(raw.head())


files: 36
rows: 900
hashes: ['murmurhash3', 'siphash24', 'splitmix64', 'xxhash64']
k values: [10, 14, 18]
d values: [100000, 1000000, 10000000]
rho values: [1, 10, 100]


,algorithm,params,mode,pairs,sample_size,pair_index,seed,estimate_merge,estimate_serial,delta_merge_serial_abs,delta_merge_serial_rel,namespace,hash,k,d,source_path,rho
0,HyperLogLog++,k=10,merge,25,10000000,0,21041998,203196,203196,0,0,merge_hpp_hom_d_100000,murmurhash3,10,100000,/home/daniele/Dev/satp-cpp/results/merge_hpp_h...,100
1,HyperLogLog++,k=10,merge,25,10000000,1,21041998,207421,207421,0,0,merge_hpp_hom_d_100000,murmurhash3,10,100000,/home/daniele/Dev/satp-cpp/results/merge_hpp_h...,100
2,HyperLogLog++,k=10,merge,25,10000000,2,21041998,206673,206673,0,0,merge_hpp_hom_d_100000,murmurhash3,10,100000,/home/daniele/Dev/satp-cpp/results/merge_hpp_h...,100
3,HyperLogLog++,k=10,merge,25,10000000,3,21041998,195505,195505,0,0,merge_hpp_hom_d_100000,murmurhash3,10,100000,/home/daniele/Dev/satp-cpp/results/merge_hpp_h...,100
4,HyperLogLog++,k=10,merge,25,10000000,4,21041998,195377,195377,0,0,merge_hpp_hom_d_100000,murmurhash3,10,100000,/home/daniele/Dev/satp-cpp/results/merge_hpp_h...,100


In [3]:
summary = (
    raw.groupby(['namespace', 'd', 'rho', 'hash', 'k'], as_index=False)
       .agg(
           pairs=('pair_index', 'count'),
           estimate_merge_mean=('estimate_merge', 'mean'),
           estimate_serial_mean=('estimate_serial', 'mean'),
           delta_abs_mean=('delta_merge_serial_abs', 'mean'),
           delta_abs_max=('delta_merge_serial_abs', 'max'),
           delta_abs_rmse=('delta_merge_serial_abs', lambda s: float(np.sqrt(np.mean(np.square(s))))),
           delta_rel_mean=('delta_merge_serial_rel', 'mean'),
           delta_rel_max=('delta_merge_serial_rel', 'max'),
           nonzero_pairs=('delta_merge_serial_abs', lambda s: int((np.abs(s.to_numpy(dtype=float)) > 0.0).sum())),
       )
)
summary['nonzero_share'] = summary['nonzero_pairs'] / summary['pairs']
summary = summary.sort_values(['rho', 'hash', 'k']).reset_index(drop=True)

summary_out = REPO / 'results' / 'merge_hpp_hom_summary.csv'
summary.to_csv(summary_out, index=False)

print('saved summary:', summary_out)
display(summary)

worst = summary.sort_values(['delta_rel_max', 'delta_abs_max'], ascending=False).head(10)
display(worst)


saved summary: /home/daniele/Dev/satp-cpp/results/merge_hpp_hom_summary.csv


,namespace,d,rho,hash,k,pairs,estimate_merge_mean,estimate_serial_mean,delta_abs_mean,delta_abs_max,delta_abs_rmse,delta_rel_mean,delta_rel_max,nonzero_pairs,nonzero_share
0,merge_hpp_hom_d_10000000,10000000,1,murmurhash3,10,25,9982066.00,9982066.00,0.0,0,0.0,0.0,0,0,0.0
1,merge_hpp_hom_d_10000000,10000000,1,murmurhash3,14,25,9974498.00,9974498.00,0.0,0,0.0,0.0,0,0,0.0
2,merge_hpp_hom_d_10000000,10000000,1,murmurhash3,18,25,10012525.00,10012525.00,0.0,0,0.0,0.0,0,0,0.0
3,merge_hpp_hom_d_10000000,10000000,1,siphash24,10,25,10461043.00,10461043.00,0.0,0,0.0,0.0,0,0,0.0
4,merge_hpp_hom_d_10000000,10000000,1,siphash24,14,25,9929434.00,9929434.00,0.0,0,0.0,0.0,0,0,0.0
5,merge_hpp_hom_d_10000000,10000000,1,siphash24,18,25,10001670.00,10001670.00,0.0,0,0.0,0.0,0,0,0.0
6,merge_hpp_hom_d_10000000,10000000,1,splitmix64,10,25,9980399.00,9980399.00,0.0,0,0.0,0.0,0,0,0.0
7,merge_hpp_hom_d_10000000,10000000,1,splitmix64,14,25,9973671.00,9973671.00,0.0,0,0.0,0.0,0,0,0.0
8,merge_hpp_hom_d_10000000,10000000,1,splitmix64,18,25,10013256.00,10013256.00,0.0,0,0.0,0.0,0,0,0.0
9,merge_hpp_hom_d_10000000,10000000,1,xxhash64,10,25,10164168.00,10164168.00,0.0,0,0.0,0.0,0,0,0.0


,namespace,d,rho,hash,k,pairs,estimate_merge_mean,estimate_serial_mean,delta_abs_mean,delta_abs_max,delta_abs_rmse,delta_rel_mean,delta_rel_max,nonzero_pairs,nonzero_share
0,merge_hpp_hom_d_10000000,10000000,1,murmurhash3,10,25,9982066.0,9982066.0,0.0,0,0.0,0.0,0,0,0.0
1,merge_hpp_hom_d_10000000,10000000,1,murmurhash3,14,25,9974498.0,9974498.0,0.0,0,0.0,0.0,0,0,0.0
2,merge_hpp_hom_d_10000000,10000000,1,murmurhash3,18,25,10012525.0,10012525.0,0.0,0,0.0,0.0,0,0,0.0
3,merge_hpp_hom_d_10000000,10000000,1,siphash24,10,25,10461043.0,10461043.0,0.0,0,0.0,0.0,0,0,0.0
4,merge_hpp_hom_d_10000000,10000000,1,siphash24,14,25,9929434.0,9929434.0,0.0,0,0.0,0.0,0,0,0.0
5,merge_hpp_hom_d_10000000,10000000,1,siphash24,18,25,10001670.0,10001670.0,0.0,0,0.0,0.0,0,0,0.0
6,merge_hpp_hom_d_10000000,10000000,1,splitmix64,10,25,9980399.0,9980399.0,0.0,0,0.0,0.0,0,0,0.0
7,merge_hpp_hom_d_10000000,10000000,1,splitmix64,14,25,9973671.0,9973671.0,0.0,0,0.0,0.0,0,0,0.0
8,merge_hpp_hom_d_10000000,10000000,1,splitmix64,18,25,10013256.0,10013256.0,0.0,0,0.0,0.0,0,0,0.0
9,merge_hpp_hom_d_10000000,10000000,1,xxhash64,10,25,10164168.0,10164168.0,0.0,0,0.0,0.0,0,0,0.0


In [4]:
def plot_heatmap_grid(df: pd.DataFrame,
                      metric: str,
                      title: str,
                      out_name: str,
                      fmt: str = '.2e',
                      cmap: str = 'viridis'):
    rho_values = sorted(df['rho'].unique().tolist())
    k_values = sorted(df['k'].unique().tolist())
    hash_values = [h for h in HASH_ORDER if h in df['hash'].unique()]

    fig, axes = plt.subplots(1, len(rho_values), figsize=(5.1 * len(rho_values), 4.4), constrained_layout=True)
    if len(rho_values) == 1:
        axes = [axes]

    vmax = float(df[metric].max())
    if vmax <= 0.0:
        vmax = 1.0

    image = None
    for ax, rho in zip(axes, rho_values):
        sub = df[df['rho'] == rho]
        pivot = (
            sub.pivot(index='hash', columns='k', values=metric)
               .reindex(index=hash_values, columns=k_values)
        )
        values = pivot.to_numpy(dtype=float)

        image = ax.imshow(values, cmap=cmap, vmin=0.0, vmax=vmax, aspect='auto')
        ax.set_title(f'rho = {rho}')
        ax.set_xlabel('k (p per HLL++)')
        ax.set_ylabel('hash')
        ax.set_xticks(np.arange(len(k_values)))
        ax.set_xticklabels(k_values)
        ax.set_yticks(np.arange(len(hash_values)))
        ax.set_yticklabels(hash_values)

        for row_idx, hash_name in enumerate(hash_values):
            for col_idx, k in enumerate(k_values):
                value = pivot.loc[hash_name, k]
                if pd.isna(value):
                    label = 'NA'
                elif fmt == 'd':
                    label = f'{int(round(value))}'
                else:
                    label = format(float(value), fmt)
                ax.text(col_idx, row_idx, label, ha='center', va='center', fontsize=9, color='black')

    fig.suptitle(title, fontsize=13)
    if image is not None:
        fig.colorbar(image, ax=axes, shrink=0.88)

    out_path = OUT_DIR / out_name
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print('saved:', out_path)
    return out_path


out_rel_max = plot_heatmap_grid(
    summary,
    metric='delta_rel_max',
    title='Merge baseline omogeneo HLL++ - delta rel max (merge vs serial)',
    out_name='merge_hpp_homogeneous_delta_rel_max_heatmap.png',
    fmt='.2e',
    cmap='magma',
)

out_nonzero = plot_heatmap_grid(
    summary,
    metric='nonzero_pairs',
    title='Merge baseline omogeneo HLL++ - numero di pair con delta non nullo',
    out_name='merge_hpp_homogeneous_nonzero_pairs_heatmap.png',
    fmt='d',
    cmap='Blues',
)


saved: /home/daniele/Dev/satp-cpp/thesis/figures/results/merge_hpp_homogeneous_delta_rel_max_heatmap.png
saved: /home/daniele/Dev/satp-cpp/thesis/figures/results/merge_hpp_homogeneous_nonzero_pairs_heatmap.png


In [5]:
summary[['rho', 'hash', 'k', 'pairs', 'delta_abs_mean', 'delta_abs_max', 'delta_rel_mean', 'delta_rel_max', 'nonzero_pairs', 'nonzero_share']] \
    .sort_values(['rho', 'hash', 'k'])


,rho,hash,k,pairs,delta_abs_mean,delta_abs_max,delta_rel_mean,delta_rel_max,nonzero_pairs,nonzero_share
0,1,murmurhash3,10,25,0.0,0,0.0,0,0,0.0
1,1,murmurhash3,14,25,0.0,0,0.0,0,0,0.0
2,1,murmurhash3,18,25,0.0,0,0.0,0,0,0.0
3,1,siphash24,10,25,0.0,0,0.0,0,0,0.0
4,1,siphash24,14,25,0.0,0,0.0,0,0,0.0
5,1,siphash24,18,25,0.0,0,0.0,0,0,0.0
6,1,splitmix64,10,25,0.0,0,0.0,0,0,0.0
7,1,splitmix64,14,25,0.0,0,0.0,0,0,0.0
8,1,splitmix64,18,25,0.0,0,0.0,0,0,0.0
9,1,xxhash64,10,25,0.0,0,0.0,0,0,0.0
